# Gold facts and marts

Use event-time tariffs and recompute percentiles from request-level measures. Keep provenance in grouping keys.


In [ ]:
if "lake_root" not in globals():
    dbutils.widgets.text("lake_root", "/Volumes/workspace/default/finops/lake")
    lake_root = dbutils.widgets.get("lake_root").rstrip("/")

from pyspark.sql import Window
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")
events = spark.read.format("delta").load(f"{lake_root}/silver/events").alias("e")
versions = spark.read.format("delta").load(f"{lake_root}/silver/price_versions").alias("p")
rates = F.when(F.col("e.data_kind") == "synthetic", "demo").otherwise("openrouter")
matches = (
    (F.col("e.model_id") == F.col("p.model_id")) & (rates == F.col("p.price_source"))
    & (F.col("e.occurred_at") >= F.col("p.valid_from"))
    & ((F.col("e.occurred_at") < F.col("p.valid_to")) | F.col("p.valid_to").isNull())
)
tariff = (F.col("e.input_tokens").cast("decimal(12,0)") * F.col("p.input_usd")
          + F.col("e.output_tokens").cast("decimal(12,0)") * F.col("p.output_usd"))
facts = events.join(versions, matches, "left").select(
    "e.*", "p.price_version_id", tariff.alias("tariff_cost_usd")
).withColumn("cost_usd", F.coalesce("charged_usd", "tariff_cost_usd")).withColumn(
    "cost_basis", F.when((F.col("data_kind") == "synthetic") & F.col("charged_usd").isNotNull(), "simulation")
    .when(F.col("charged_usd").isNotNull(), "reported")
    .when(F.col("tariff_cost_usd").isNotNull(), "catalog_estimate").otherwise("unknown")
)
daily = facts.groupBy("event_date", "tenant_id", "workflow", "model_id", "data_kind", "source", "cost_basis").agg(
    F.count("*").alias("runs"), F.count("cost_usd").alias("priced_runs"),
    F.count_if(F.col("status") == "ok").alias("successful_runs"),
    F.sum("input_tokens").alias("input_tokens"), F.sum("output_tokens").alias("output_tokens"),
    F.sum("cost_usd").alias("cost_usd"), F.percentile("latency_ms", 0.95).alias("p95_latency_ms")
)
experiment = facts.filter(F.col("experiment_id").isNotNull()).groupBy("experiment_id", "data_kind", "arm").agg(
    F.count("*").alias("assigned"), F.count("cost_usd").alias("priced_runs"),
    F.count_if(F.col("status") == "ok").alias("successful_runs"),
    F.count_if("fallback").alias("fallbacks"),
    F.count("ground_truth").alias("labels_available"),
    F.count_if(F.col("prediction") == F.col("ground_truth")).alias("correct"),
    F.avg("cost_usd").alias("mean_cost_usd"), F.percentile("latency_ms", 0.95).alias("p95_latency_ms")
)
scores = spark.read.format("delta").load(f"{lake_root}/silver/scores")
ranking = Window.partitionBy("source", "model_id", "benchmark", "revision").orderBy(F.desc("observed_at"))
latest_scores = scores.withColumn("ranking", F.row_number().over(ranking)).filter("ranking = 1").drop("ranking")
tables = {
    "fct_llm_runs": facts,
    "dim_models_scd2": versions.withColumn("is_current", F.col("valid_to").isNull()),
    "dim_tenants": facts.groupBy("tenant_id").agg(F.min("event_date").alias("first_seen_date"),
                                                 F.max("event_date").alias("last_seen_date")),
    "dim_models": facts.select("model_id").union(versions.select("model_id")).distinct(),
    "mart_finops_daily": daily,
    "mart_ab_routing": experiment,
    "mart_model_benchmarks": latest_scores,
}
for name, frame in tables.items():
    frame.write.format("delta").mode("overwrite").save(f"{lake_root}/gold/{name}")
    print(name, spark.read.format("delta").load(f"{lake_root}/gold/{name}").count())
